In [2]:
# 1. Framework initialization
!pip install -q mediapipe opencv-python gradio

import os
import cv2
import numpy as np
import gradio as gr
import urllib.request

# Clean file footprint to prevent corrupt reads
if os.path.exists('face_landmarker.task'):
    try: os.remove('face_landmarker.task')
    except: pass

print("Direct storage server verification architecture initialize ho raha hai...")
# Exact official Google API endpoint framework download trigger
urllib.request.urlretrieve(
    "https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task",
    "face_landmarker.task"
)
print("Download 100% Verified and Saved!")

# 3. Model setup logic
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

base_options = python.BaseOptions(model_asset_path='face_landmarker.task')
options = vision.FaceLandmarkerOptions(base_options=base_options, num_faces=1)
detector = vision.FaceLandmarker.create_from_options(options)

# MediaPipe structural mesh points mapping array config
LEFT_EYE_INDICES  = [33, 160, 158, 133, 153, 144]
RIGHT_EYE_INDICES = [362, 385, 387, 263, 373, 380]

def get_ear(landmarks, eye_indices, img_w, img_h):
    points = []
    # Nested MediaPipe extraction format validation
    target_landmarks = landmarks[0] if isinstance(landmarks[0], list) else landmarks

    for idx in eye_indices:
        lm = target_landmarks[idx]
        points.append(np.array([lm.x * img_w, lm.y * img_h]))

    p2_p6 = np.linalg.norm(points[1] - points[5])
    p3_p4 = np.linalg.norm(points[2] - points[4])
    p1_p4_dist = np.linalg.norm(points[0] - points[3])

    return (p2_p6 + p3_p4) / (2.0 * p1_p4_dist)

state = {"COUNTER": 0}
EAR_THRESHOLD = 0.22

def detect_drowsiness(frame):
    if frame is None:
        return None

    frame = cv2.flip(frame, 1)
    h, w, _ = frame.shape

    import mediapipe as mp
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame)
    detection_result = detector.detect(mp_image)

    if detection_result and detection_result.face_landmarks and len(detection_result.face_landmarks) > 0:
        raw_landmarks = detection_result.face_landmarks[0]

        left_ear = get_ear(raw_landmarks, LEFT_EYE_INDICES, w, h)
        right_ear = get_ear(raw_landmarks, RIGHT_EYE_INDICES, w, h)
        avg_ear = (left_ear + right_ear) / 2.0

        # Output printing (Bright green indicator)
        cv2.putText(frame, f"EAR: {avg_ear:.2f}", (30, 60), cv2.FONT_HERSHEY_SIMPLEX, 1.3, (0, 255, 0), 3)

        if avg_ear < EAR_THRESHOLD:
            state["COUNTER"] += 1
            if state["COUNTER"] >= 5:
                cv2.putText(frame, "!!! DROWSINESS ALERT !!!", (30, 130), cv2.FONT_HERSHEY_SIMPLEX, 1.4, (0, 0, 255), 4)
        else:
            state["COUNTER"] = 0
    else:
        cv2.putText(frame, "Looking for Face...", (30, 60), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 165, 255), 2)

    return frame

# 4. Gradio Interface Layout Render
interface = gr.Interface(
    fn=detect_drowsiness,
    inputs=gr.Image(sources=["webcam"], streaming=True),
    outputs="image",
    live=True,
    title="Driver Drowsiness Detector"
)

interface.launch(share=True, debug=True)


Direct storage server verification architecture initialize ho raha hai...
Download 100% Verified and Saved!
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://d3589bd2dff957a167.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://d3589bd2dff957a167.gradio.live
